In [1]:
import re
import pandas as pd

In [2]:
with open("support_logs_2025-07-01.log", "r", encoding="utf-8") as f:
    content = f.read()
len(content)

32938

In [3]:
# lines = list(map(str.strip, content.split("---")))
# lines = list(filter(None, lines))
lines = [x.strip() for x in content.split("---") if x]
lines[-1]

'2025-07-01 14:10:00 [INFO] careplus.support.GenericService - TicketID=TCK0701029 SessionID=sess_TCK0701029\nIP=178.77.232.8 | ResponseTime=214ms | CPU=33.47% | EventType=generic_event | Error=false\nUserAgent="curl/7.68.0"\nMessage=" event for TCK0701029"\nDebug="ℹ️ Logged for monitoring"\nTraceID=None'

In [4]:
log_pattern = re.compile(
    r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(?P<log_level>[A-Za-z0-9_]+)\] "
    r"(?P<component>[^\s]+) - TicketID=(?P<ticket_id>\S+) SessionID=(?P<session_id>\S+)\s+"
    r"IP=(?P<ip>.*?) \| ResponseTime=(?P<response_time>-?\d+)ms \| CPU=(?P<cpu>[\d.]+)% \| EventType=(?P<event_type>.*?) \| Error=(?P<error>\w+)\s+"
    r"UserAgent=\"(?P<user_agent>.*?)\"\s+"
    r"Message=\"(?P<message>.*?)\"\s+"
    r"Debug=\"(?P<debug>.*?)\"\s+"
    r"TraceID=(?P<trace_id>\S+)"
)
parsed_entries = []
for entry in lines:
    match = log_pattern.search(entry)
    if match:
        parsed_entries.append(match.groupdict())
        
parsed_entries[0]


{'timestamp': '2025-07-01 00:21:00',
 'log_level': 'INF0',
 'component': 'careplus.support.GenericService',
 'ticket_id': 'TCK0701000',
 'session_id': 'sess_TCK0701000',
 'ip': '60.130.155.7',
 'response_time': '1269',
 'cpu': '27.64',
 'event_type': 'generic_event',
 'error': 'false',
 'user_agent': 'PostmanRuntime/7.32.2',
 'message': ' event for TCK0701000',
 'debug': 'ℹ️ Logged for monitoring',
 'trace_id': 'None'}

In [35]:
df = pd.DataFrame(parsed_entries)
df.drop("trace_id", axis=1, inplace=True)
df.head()

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
0,2025-07-01 00:21:00,INF0,careplus.support.GenericService,TCK0701000,sess_TCK0701000,60.130.155.7,1269,27.64,generic_event,false,PostmanRuntime/7.32.2,event for TCK0701000,ℹ️ Logged for monitoring
1,2025-07-01 00:41:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,58.36.189.27,1505,57.24,generic_event,false,Mobile-Safari/537.36,event for TCK0701000,ℹ️ Logged for monitoring
2,2025-07-01 01:44:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,181.18.12.170,586,78.43,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring
3,2025-07-01 01:49:00,DEBUG,careplus.support.GenericService,TCK0701001,sess_TCK0701001,163.214.94.42,878,63.61,generic_event,false,curl/7.68.0,event for TCK0701001,ℹ️ Logged for monitoring
4,2025-07-01 01:50:00,INFO,careplus.support.GenericService,TCK0701000,sess_TCK0701000,155.68.207.12,1614,87.85,generic_event,false,Python-urllib/3.9,event for TCK0701000,ℹ️ Logged for monitoring


In [36]:
df = df.astype({"response_time": "int", "cpu": "float"})
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
df["error"] = df["error"].str.lower().map({"false": False, "true": True})
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      105 non-null    datetime64[us]
 1   log_level      105 non-null    str           
 2   component      105 non-null    str           
 3   ticket_id      105 non-null    str           
 4   session_id     105 non-null    str           
 5   ip             105 non-null    str           
 6   response_time  105 non-null    int64         
 7   cpu            105 non-null    float64       
 8   event_type     105 non-null    str           
 9   error          105 non-null    bool          
 10  user_agent     105 non-null    str           
 11  message        105 non-null    str           
 12  debug          105 non-null    str           
dtypes: bool(1), datetime64[us](1), float64(1), int64(1), str(9)
memory usage: 10.1 KB


In [37]:
df.drop(df[df["response_time"] < 0].index, axis=0, inplace=True)
df.describe()

,timestamp,response_time,cpu
count,98,98.000000,98.000000
mean,2025-07-01 08:24:49.591836,1006.500000,54.821633
min,2025-07-01 00:21:00,126.000000,13.350000
25%,2025-07-01 05:39:15,653.250000,34.710000
50%,2025-07-01 09:15:00,1089.000000,59.455000
75%,2025-07-01 11:33:00,1327.750000,73.242500
max,2025-07-01 14:10:00,1792.000000,89.970000
std,NaN,447.808944,22.185337


In [42]:
df["log_level"] = df["log_level"].replace({"INF0": "INFO", "DEBG": "DEBUG", "warnING": "WARNING"})
df.drop_duplicates(inplace=True)
df[df.duplicated()]

,timestamp,log_level,component,ticket_id,session_id,ip,response_time,cpu,event_type,error,user_agent,message,debug
